# Assignment 4 — Distributed dense matrix multiplication (PySpark)

Analysis of the campaign in `data/results/configs4.csv` (40 configurations, all `ok`):

* **Strong scaling** — fixed `n ∈ {2000, 4000}`, `p ∈ {1, 2, 4, 8}` row blocks / workers: `S(p) = T(1)/T(p)`, `E(p) = S(p)/p`.
* **Weak scaling** — `(p, n) = (1,2000), (2,2500), (4,3200), (8,4000)`, i.e. `n = 2000·p^(1/3)` rounded, so that `n³/p` (work per worker) stays constant.
* **Size sweep** — `p = 4`, `n ∈ {500, 1000, 2000, 4000}`.
* **References** — single-process numpy/BLAS with the *same* kernel, plus the Assignment 3 sequential C and OpenMP/SIMD C kernels.
* **Overhead** — `t_e2e` split into the in-worker matmul and everything Spark adds around it.

Timing boundary: `t_e2e = t_stage_a + t_action_job + t_assemble`, median of 3 repetitions after 1 warm-up, under the *High performance* power plan. `t_startup` and `t_replicate` are one-time per application and reported separately.


In [1]:
from pathlib import Path
import csv
import statistics as st

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "results").exists():
    ROOT = ROOT.parent
RES = ROOT / "data" / "results"
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

cfgs = list(csv.DictReader((RES / "configs4.csv").open(encoding="utf-8")))
raw = list(csv.DictReader((RES / "raw_timings4.csv").open(encoding="utf-8")))


def fnum(row, key, default=None):
    v = row.get(key, "")
    return float(v) if v not in ("", None) else default


ok = [c for c in cfgs if c["status"] == "ok"]
spark = [c for c in ok if c["variant"] == "spark"]

STRONG_N = [2000, 4000]
PS = [1, 2, 4, 8]
WEAK = [(1, 2000), (2, 2500), (4, 3200), (8, 4000)]
SIZE_N = [500, 1000, 2000, 4000]
SIZE_P = 4


def spark_row(exp, n, p):
    for c in ok:
        if (c["experiment"] == exp and c["variant"] == "spark"
                and int(c["n"]) == n and int(c["p"]) == p):
            return c
    raise KeyError((exp, n, p))


def baseline(variant, n, p=None):
    for c in ok:
        if (c["experiment"] == "baselines" and c["variant"] == variant
                and int(c["n"]) == n
                and (p is None or int(c["p"]) == p)):
            return fnum(c, "measured_median_s")
    raise KeyError((variant, n, p))


def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"wrote {name}.pdf/png")


print(f"configs={len(cfgs)} ok={len(ok)} spark={len(spark)} "
      f"raw_repetitions={len(raw)}")
print("experiments:", sorted({c["experiment"] for c in ok}))


configs=40 ok=40 spark=16 raw_repetitions=168
experiments: ['baselines', 'size', 'strong', 'weak']


## Validation

Every distributed configuration and every numpy reference is checked element-wise against the sequential C result (`build/ref/C_n_seq.bin`) with `rtol=1e-9`, `atol=1e-12`. Summation order differs between BLAS and the `ikj` C kernel, so the criterion is tolerance-based rather than bitwise.

`shuffle = 0` for every Spark configuration confirms the design is genuinely a no-shuffle plan: A is staged once, B is replicated with `sc.addFile`, and partial C blocks are collected.


In [2]:
val = [c for c in ok if c["valid"]]
bad = [c for c in val if c["valid"] != "true"]
diffs = [fnum(c, "max_abs_diff") for c in val if fnum(c, "max_abs_diff")]
shuf = ({c["shuffle_write_bytes"] for c in spark}
        | {c["shuffle_read_bytes"] for c in spark})

print(f"validated configs: {len(val)}/{len(ok)}  (16 Spark + 4 numpy)")
print("tolerance failures:", bad or "none")
print(f"max_abs_diff over all validated runs: {max(diffs):.3e}")
print("shuffle bytes across all Spark configs:", sorted(shuf))
assert not bad and shuf == {"0"}, "validation or shuffle invariant broken"


validated configs: 20/40  (16 Spark + 4 numpy)
tolerance failures: none
max_abs_diff over all validated runs: 6.963e-13
shuffle bytes across all Spark configs: ['0']


## Strong scaling

Fixed problem size, growing worker count. `T_p` is the median end-to-end time of one multiplication; `matmul` is the pure BLAS call measured *inside* the worker, so `overhead = T_p - matmul` is everything Spark adds (task dispatch, Python-worker respawn, transfer of A and C).


In [3]:
strong = []
for n in STRONG_N:
    t1 = fnum(spark_row("strong", n, 1), "measured_median_s")
    for p in PS:
        c = spark_row("strong", n, p)
        t = fnum(c, "measured_median_s")
        mm = (fnum(c, "matmul_ms_max") or 0.0) / 1000.0
        strong.append(dict(
            n=n, p=p, t_e2e_s=t, speedup=t1 / t, efficiency=t1 / t / p,
            matmul_s=mm, overhead_s=t - mm, overhead_pct=100.0 * (t - mm) / t,
            stage_a_s=fnum(c, "stage_a_s"), action_job_s=fnum(c, "action_job_s"),
            assemble_s=fnum(c, "assemble_s"),
            startup_s=fnum(c, "startup_s"), replicate_s=fnum(c, "replicate_s"),
            n_tasks=int(c["n_tasks"]), n_executors=int(c["n_executors"]),
            app_id=c["app_id"]))

hdr = (f"{'n':>5} {'p':>3} {'T_e2e(s)':>9} {'S(p)':>6} {'E(p)':>6} "
       f"{'matmul':>8} {'overhead':>9} {'ov%':>6} {'startup':>8} {'repl':>6}")
print(hdr)
for r in strong:
    print(f"{r['n']:5d} {r['p']:3d} {r['t_e2e_s']:9.4f} {r['speedup']:6.2f} "
          f"{r['efficiency']:6.3f} {r['matmul_s']:8.3f} "
          f"{r['overhead_s']:9.3f} {r['overhead_pct']:6.1f} "
          f"{r['startup_s']:8.2f} {r['replicate_s']:6.2f}")

with (RES / "summary_strong.csv").open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(strong[0]))
    w.writeheader()
    w.writerows(strong)
print("wrote summary_strong.csv")


    n   p  T_e2e(s)   S(p)   E(p)   matmul  overhead    ov%  startup   repl
 2000   1    1.7031   1.00  1.000    0.271     1.433   84.1     8.96   2.12
 2000   2    1.9347   0.88  0.440    0.204     1.731   89.5    11.82   2.47
 2000   4    2.4044   0.71  0.177    0.268     2.136   88.8    14.27   2.88
 2000   8    3.5362   0.48  0.060    0.284     3.252   92.0    17.44   6.17
 4000   1    5.5437   1.00  1.000    2.728     2.815   50.8    11.71   3.71
 4000   2    4.9338   1.12  0.562    1.684     3.250   65.9    11.83   2.75
 4000   4    5.2604   1.05  0.263    1.756     3.505   66.6    13.47   2.94
 4000   8    9.3778   0.59  0.074    2.568     6.810   72.6    13.32   8.58
wrote summary_strong.csv


In [4]:
fig, ax = plt.subplots(figsize=(6.8, 4.4))
for n in STRONG_N:
    rows_n = [r for r in strong if r["n"] == n]
    ax.plot(PS, [r["t_e2e_s"] for r in rows_n], marker="o", linewidth=1.8,
            label=f"Spark end-to-end n={n}")
    ax.plot(PS, [r["matmul_s"] for r in rows_n], marker="s", linestyle="--",
            alpha=0.65, label=f"matmul alone n={n}")
    t = baseline("numpy", n)
    ax.axhline(t, linewidth=1.0, linestyle=":", alpha=0.85,
               label=f"numpy 1 process n={n} ({t:.3f} s)")
ax.set_xscale("log", base=2)
ax.set_xticks(PS, labels=[str(p) for p in PS])
ax.set_yscale("log")
ax.set_xlabel("workers p (row blocks of A)")
ax.set_ylabel("time per multiplication (s)")
ax.set_title("Strong scaling: end-to-end Spark time vs the matmul alone\n"
             "(dotted: single-process numpy/BLAS reference)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.16))
fig.subplots_adjust(bottom=0.30)
save(fig, "fig1_strong")


wrote fig1_strong.pdf/png


In [5]:
fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot([1, 8], [1, 8], linestyle="--", color="black", linewidth=1.1,
        label="ideal S(p) = p")
ax.axhline(1.0, color="tab:red", linewidth=1.0, linestyle=":",
           label="S(p) = 1 (no gain)")
for n in STRONG_N:
    rows_n = [r for r in strong if r["n"] == n]
    ax.plot(PS, [r["speedup"] for r in rows_n], marker="o", linewidth=1.8,
            label=f"n={n}")
    best = max(rows_n, key=lambda r: r["speedup"])
    if best["p"] != 1:
        ax.annotate(f"best {best['speedup']:.2f}x at p={best['p']}",
                    (best["p"], best["speedup"]), textcoords="offset points",
                    xytext=(10, 10), fontsize=8)
ax.set_xscale("log", base=2)
ax.set_xticks(PS, labels=[str(p) for p in PS])
ax.set_xlabel("workers p")
ax.set_ylabel("speedup S(p) = T(1) / T(p)")
ax.set_title("Strong-scaling speedup (median of 3 repetitions)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
save(fig, "fig2_speedup")


wrote fig2_speedup.pdf/png


## Weak scaling

Work per worker is held constant: `n = 2000·p^(1/3)` rounded, so `n³/p ≈ 8·10⁹` for every point. Perfect weak scaling keeps `T_p` flat, i.e. `ratio_to_p1 = 1`.


In [6]:
weak = []
t1 = fnum(spark_row("weak", 2000, 1), "measured_median_s")
for p, n in WEAK:
    c = spark_row("weak", n, p)
    t = fnum(c, "measured_median_s")
    mm = (fnum(c, "matmul_ms_max") or 0.0) / 1000.0
    weak.append(dict(
        p=p, n=n, work_per_worker=round(n ** 3 / p),
        t_e2e_s=t, ratio_to_p1=t / t1, weak_efficiency=t1 / t,
        matmul_s=mm, overhead_s=t - mm, overhead_pct=100.0 * (t - mm) / t,
        startup_s=fnum(c, "startup_s"), replicate_s=fnum(c, "replicate_s"),
        app_id=c["app_id"]))

print(f"{'p':>3} {'n':>5} {'n^3/p':>10} {'T_e2e(s)':>9} {'T_p/T_1':>8} "
      f"{'weak E':>7} {'matmul':>8} {'overhead':>9} {'ov%':>6}")
for r in weak:
    print(f"{r['p']:3d} {r['n']:5d} {r['work_per_worker']:10d} "
          f"{r['t_e2e_s']:9.4f} {r['ratio_to_p1']:8.3f} "
          f"{r['weak_efficiency']:7.3f} {r['matmul_s']:8.3f} "
          f"{r['overhead_s']:9.3f} {r['overhead_pct']:6.1f}")

with (RES / "summary_weak.csv").open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(weak[0]))
    w.writeheader()
    w.writerows(weak)
print("wrote summary_weak.csv")


  p     n      n^3/p  T_e2e(s)  T_p/T_1  weak E   matmul  overhead    ov%
  1  2000 8000000000    2.0858    1.000   1.000    0.308     1.778   85.2
  2  2500 7812500000    2.2615    1.084   0.922    0.381     1.881   83.2
  4  3200 8192000000    3.5427    1.698   0.589    1.040     2.503   70.6
  8  4000 8000000000    7.7475    3.714   0.269    1.940     5.807   75.0
wrote summary_weak.csv


In [7]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.0))

ps = [r["p"] for r in weak]
ax1.plot(ps, [r["t_e2e_s"] for r in weak], marker="o", linewidth=1.8,
         label="end-to-end T_p")
ax1.plot(ps, [r["matmul_s"] for r in weak], marker="s", linestyle="--",
         alpha=0.7, label="matmul alone")
ax1.axhline(t1, color="tab:red", linewidth=1.0, linestyle=":",
            label="perfect weak scaling (T_p = T_1)")
ax1.set_xscale("log", base=2)
ax1.set_xticks(ps, labels=[str(p) for p in ps])
ax1.set_xlabel("workers p")
ax1.set_ylabel("time per multiplication (s)")
ax1.set_title("Weak scaling (n^3/p constant)")
ax1.grid(True, which="both", alpha=0.3)
ax1.legend(fontsize=8)

ax2.plot(ps, [r["ratio_to_p1"] for r in weak], marker="o", linewidth=1.8,
         label="T_p / T_1")
ax2.plot(ps, [r["weak_efficiency"] for r in weak], marker="s", linewidth=1.8,
         label="weak efficiency T_1 / T_p")
ax2.axhline(1.0, color="black", linewidth=1.0, linestyle="--", alpha=0.6)
ax2.set_xscale("log", base=2)
ax2.set_xticks(ps, labels=[str(p) for p in ps])
ax2.set_xlabel("workers p")
ax2.set_ylabel("ratio")
ax2.set_title("Weak-scaling degradation")
ax2.grid(True, which="both", alpha=0.3)
ax2.legend(fontsize=8)

fig.tight_layout()
save(fig, "fig3_weak")


wrote fig3_weak.pdf/png


## Problem-size sweep at p = 4

Shows the fixed cost the framework imposes: at small `n` the matmul is a rounding error and `T_p` is essentially pure overhead.


In [8]:
size = []
for n in SIZE_N:
    c = spark_row("size", n, SIZE_P)
    t = fnum(c, "measured_median_s")
    mm = (fnum(c, "matmul_ms_max") or 0.0) / 1000.0
    size.append(dict(
        n=n, p=SIZE_P, t_e2e_s=t, matmul_s=mm, overhead_s=t - mm,
        overhead_pct=100.0 * (t - mm) / t,
        matmul_share_pct=100.0 * mm / t,
        numpy_s=baseline("numpy", n),
        c_seq_s=baseline("c_seq", n),
        startup_s=fnum(c, "startup_s"), replicate_s=fnum(c, "replicate_s"),
        app_id=c["app_id"]))

print(f"{'n':>5} {'T_e2e(s)':>9} {'matmul':>8} {'share%':>7} "
      f"{'overhead':>9} {'ov%':>6} {'numpy':>8} {'c_seq':>8}")
for r in size:
    print(f"{r['n']:5d} {r['t_e2e_s']:9.4f} {r['matmul_s']:8.3f} "
          f"{r['matmul_share_pct']:7.1f} {r['overhead_s']:9.3f} "
          f"{r['overhead_pct']:6.1f} {r['numpy_s']:8.3f} {r['c_seq_s']:8.3f}")

with (RES / "summary_size.csv").open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(size[0]))
    w.writeheader()
    w.writerows(size)
print("wrote summary_size.csv")


    n  T_e2e(s)   matmul  share%  overhead    ov%    numpy    c_seq
  500    1.8410    0.005     0.3     1.836   99.7    0.007    0.018
 1000    1.9343    0.018     0.9     1.916   99.1    0.063    0.186
 2000    2.6518    0.195     7.4     2.457   92.6    0.348    3.321
 4000    5.1452    1.753    34.1     3.392   65.9    2.604   30.071
wrote summary_size.csv


In [9]:
fig, ax = plt.subplots(figsize=(6.8, 4.4))
ns = [r["n"] for r in size]
ax.plot(ns, [r["t_e2e_s"] for r in size], marker="o", linewidth=1.8,
        label="Spark end-to-end (p=4)")
ax.plot(ns, [r["matmul_s"] for r in size], marker="s", linestyle="--",
        alpha=0.75, label="matmul alone (p=4)")
ax.plot(ns, [r["overhead_s"] for r in size], marker="^", linestyle="-.",
        alpha=0.85, label="Spark overhead = T - matmul")
ax.plot(ns, [r["numpy_s"] for r in size], marker="d", linestyle=":",
        label="numpy 1 process, 1 thread")
ax.plot(ns, [r["c_seq_s"] for r in size], marker="x", linestyle=":",
        alpha=0.8, label="C sequential (Assignment 3)")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(ns, labels=[str(n) for n in ns])
ax.set_xlabel("matrix order n")
ax.set_ylabel("time per multiplication (s)")
ax.set_title("Problem-size sweep at p = 4 workers\n"
             "(overhead is nearly flat: it does not shrink with n)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
save(fig, "fig4_size")


wrote fig4_size.pdf/png


## Where the time goes

`overhead = T_e2e - matmul` is the cost Spark adds around the kernel: staging A on the driver, dispatching the task, **respawning a Python worker** (see limitations), transferring A and C, and assembling the result. The one-time `t_startup` / `t_replicate` are shown separately because they are paid once per application, not once per multiplication.


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.2), sharey=False)
for ax, n in zip(axes, STRONG_N):
    rows_n = [r for r in strong if r["n"] == n]
    ps = [r["p"] for r in rows_n]
    mm = [r["matmul_s"] for r in rows_n]
    ov = [r["overhead_s"] for r in rows_n]
    ax.bar(ps, mm, width=0.6, label="matmul (in-worker BLAS)", color="tab:blue")
    ax.bar(ps, ov, width=0.6, bottom=mm, label="Spark overhead",
           color="tab:orange")
    for x, r in zip(ps, rows_n):
        ax.text(x, r["t_e2e_s"], f" {r['overhead_pct']:.0f}% ov",
                ha="center", va="bottom", fontsize=7.5)
    ax.set_xscale("log", base=2)
    ax.set_xticks(ps, labels=[str(p) for p in ps])
    ax.set_xlabel("workers p")
    ax.set_title(f"n = {n}   (one-time startup+replicate: "
                 f"{rows_n[0]['startup_s'] + rows_n[0]['replicate_s']:.1f} s "
                 f"at p=1)")
    ax.grid(True, axis="y", alpha=0.3)
axes[0].set_ylabel("seconds per multiplication")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("End-to-end time per multiplication, decomposed", y=1.00)
fig.tight_layout()
save(fig, "fig5_overhead")


wrote fig5_overhead.pdf/png


## Comparison against the references

`numpy` is the same BLAS kernel in one process with one thread — it isolates the cost of *distribution* from the cost of the *kernel*. The Assignment 3 C kernels are shown too, but they are a different (much slower) kernel, so they are context rather than a like-for-like baseline.


In [11]:
compare = []
for n in STRONG_N:
    t_spark = {p: fnum(spark_row("strong", n, p), "measured_median_s")
               for p in PS}
    entries = [("numpy BLAS, 1 process, 1 thread", baseline("numpy", n))]
    entries.append(("C sequential (Assignment 3)", baseline("c_seq", n)))
    for p in PS:
        entries.append((f"C OpenMP, {p} thread(s)", baseline("c_omp", n, p)))
    for p in PS:
        entries.append((f"Spark, {p} worker(s)", t_spark[p]))
    best_spark = min(t_spark.values())
    best_p = min(t_spark, key=t_spark.get)
    for label, t in entries:
        compare.append(dict(
            n=n, implementation=label, t_e2e_s=t,
            speedup_vs_numpy=baseline("numpy", n) / t,
            speedup_vs_best_spark=best_spark / t,
            is_best_spark=(label == f"Spark, {best_p} worker(s)")))


def rel(ratio):
    """Describe a time ratio against numpy in plain words."""
    return (f"{1 / ratio:.2f}x slower than numpy" if ratio < 1
            else f"{ratio:.2f}x faster than numpy")


for n in STRONG_N:
    print(f"--- n = {n} ---")
    for r in [x for x in compare if x["n"] == n]:
        print(f"  {r['implementation']:<34} {r['t_e2e_s']:8.4f} s   "
              f"{rel(r['speedup_vs_numpy'])}")
    bs = min((x for x in compare if x["n"] == n
              and x["implementation"].startswith("Spark")),
             key=lambda x: x["t_e2e_s"])
    print(f"  best Spark: {bs['implementation']} -> {bs['t_e2e_s']:.4f} s "
          f"({rel(bs['speedup_vs_numpy'])})")

with (RES / "summary_compare.csv").open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=list(compare[0]))
    w.writeheader()
    w.writerows(compare)
print("wrote summary_compare.csv")


--- n = 2000 ---
  numpy BLAS, 1 process, 1 thread      0.3480 s   1.00x faster than numpy
  C sequential (Assignment 3)          3.3214 s   9.54x slower than numpy
  C OpenMP, 1 thread(s)                3.3236 s   9.55x slower than numpy
  C OpenMP, 2 thread(s)                2.2054 s   6.34x slower than numpy
  C OpenMP, 4 thread(s)                1.5182 s   4.36x slower than numpy
  C OpenMP, 8 thread(s)                1.2530 s   3.60x slower than numpy
  Spark, 1 worker(s)                   1.7031 s   4.89x slower than numpy
  Spark, 2 worker(s)                   1.9347 s   5.56x slower than numpy
  Spark, 4 worker(s)                   2.4044 s   6.91x slower than numpy
  Spark, 8 worker(s)                   3.5362 s   10.16x slower than numpy
  best Spark: Spark, 1 worker(s) -> 1.7031 s (4.89x slower than numpy)
--- n = 4000 ---
  numpy BLAS, 1 process, 1 thread      2.6037 s   1.00x faster than numpy
  C sequential (Assignment 3)         30.0711 s   11.55x slower than numpy
  C O

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 5.0))
for ax, n in zip(axes, STRONG_N):
    rows_n = [x for x in compare if x["n"] == n]
    labels = [r["implementation"] for r in rows_n]
    vals = [r["t_e2e_s"] for r in rows_n]
    colors = []
    for lab in labels:
        if lab.startswith("numpy"):
            colors.append("tab:green")
        elif lab.startswith("C seq"):
            colors.append("tab:red")
        elif lab.startswith("C OpenMP"):
            colors.append("tab:orange")
        else:
            colors.append("tab:blue")
    ypos = range(len(labels))
    ax.barh(list(ypos), vals, color=colors, alpha=0.85)
    ax.set_yticks(list(ypos), labels=labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xscale("log")
    ax.set_xlabel("time per multiplication (s)")
    ax.set_title(f"n = {n}")
    ax.grid(True, axis="x", alpha=0.3)
    for y, v in zip(ypos, vals):
        ax.text(v * 1.05, y, f"{v:.3f}", va="center", fontsize=7.5)
fig.suptitle("Distributed implementation vs single-process references", y=1.00)
fig.tight_layout()
save(fig, "fig6_compare")


wrote fig6_compare.pdf/png


## Outputs

| File | Contents |
|---|---|
| `data/results/summary_strong.csv` | `n, p, T_e2e, S(p), E(p), matmul, overhead, startup, replicate` |
| `data/results/summary_weak.csv` | `p, n, n³/p, T_p, T_p/T_1, weak efficiency, overhead` |
| `data/results/summary_size.csv` | `n` at `p=4`: `T_e2e`, matmul share, overhead, numpy and C references |
| `data/results/summary_compare.csv` | every implementation at `n ∈ {2000, 4000}` with ratios |
| `reports/figures/fig1_strong` … `fig6_compare` | PDF (report) + PNG (preview) |
